In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from scipy.stats import pearsonr

# === CONFIGURATION ===
root_dir = 'COO-Decon-Noise'
results_dir = os.path.join(root_dir, 'Results-Pearson-PredictedVSActual')
os.makedirs(results_dir, exist_ok=True)

method_map = ['BayesPrism', 'nuSVR', 'ReDeconv', 'CIBERSORTx', 'MuSiC', 'NNLS', 'QP']

# Map methods → proportion files
method_to_prop = {
    'BayesPrism': 'TSP-BDa_Inner_Random_v2C_New_All-Proportions.txt',
    'NNLS': 'TSP-BDa_Inner_Random_v2C_New_All-Proportions.txt',
    'QP': 'TSP-BDa_Inner_Random_v2C_New_All-Proportions.txt',
    'CIBERSORTx': 'TSP-BDa_Outer_Random_v2C_New_All-Proportions.txt',
    'nuSVR': 'TSP-BDa_Outer_Random_v2C_New_All-Proportions.txt',
    'MuSiC': 'TSP-HBA_Inner_Random_v2C_New_All-Proportions.txt',
    'ReDeconv': 'TSP-HBA_Inner_Random_v2C_New_All-Proportions.txt'
}

# === FIND ALL *_modified.txt FILES ===
all_files = sorted(glob(os.path.join(root_dir, "**/*_modified.txt"), recursive=True))
print(f"Found {len(all_files)} deconvolution result files.\n")

all_results = []

# === MAIN LOOP ===
for file_path in all_files:
    try:
        filename = os.path.basename(file_path)
        dir_path = os.path.dirname(file_path)
        dirname = os.path.basename(dir_path)

        # Extract noise level from directory name
        noise_level = None
        parts = dirname.split('-')
        noise_token = [p for p in parts if p.startswith("Noisy_phi")]
        if noise_token:
            noise_level = noise_token[0].replace("Noisy_phi", "").replace("_All", "")
        else:
            noise_level = "Unknown"

        # Identify method
        raw_method = filename.replace('_modified.txt', '')
        method = next((m for m in method_map if m in raw_method), None)
        if method is None:
            print(f" Could not determine method for {file_path}")
            continue

        # Get corresponding proportion file
        prop_file = method_to_prop.get(method)
        if not prop_file:
            print(f" No proportion file mapping for {method}")
            continue

        prop_path = os.path.join(root_dir, prop_file)
        if not os.path.exists(prop_path):
            print(f" Proportions file missing: {prop_path}")
            continue

        # === LOAD AND NORMALIZE GROUND TRUTH ===
        prop_df = pd.read_csv(prop_path, sep='\t', index_col=0)
        if "TotalCells" in prop_df.columns:
            prop_df.drop(columns=["TotalCells"], inplace=True)
        prop_df = prop_df.div(prop_df.sum(axis=1), axis=0).multiply(100).round(5)

        # === LOAD DECON RESULTS ===
        decon_df = pd.read_csv(file_path, sep='\t', index_col=0)
        decon_df.index = decon_df.index.str.replace(r'_phi\d+\.?\d*(?:e[+-]?\d+)?$', '', regex=True)

        # Align columns and samples
        common_cols = prop_df.columns.intersection(decon_df.columns)
        if common_cols.empty:
            print(f" No overlapping columns in {file_path}")
            continue
        common_samples = prop_df.index.intersection(decon_df.index)
        if common_samples.empty:
            print(f" No overlapping samples in {file_path}")
            continue

        true_vals_df = prop_df.loc[common_samples, common_cols]
        pred_vals_df = decon_df.loc[common_samples, common_cols]

        # Normalize predicted values
        pred_vals_df = pred_vals_df.div(pred_vals_df.sum(axis=1), axis=0).multiply(100).round(5)

        # Flatten and filter zero-zero pairs
        true_vals_flat = true_vals_df.values.flatten()
        pred_vals_flat = pred_vals_df.values.flatten()
        mask_nonzero = ~((true_vals_flat == 0) & (pred_vals_flat == 0))
        true_vals_flat = true_vals_flat[mask_nonzero]
        pred_vals_flat = pred_vals_flat[mask_nonzero]

        # Pearson correlation
        r, p = pearsonr(true_vals_flat, pred_vals_flat)
        r2 = r ** 2
        print(f"{dirname} | {method} | Noise={noise_level} — Pearson r = {r:.4f}, p = {p:.2e}")

        all_results.append([prop_file, dirname, method, noise_level, r, r2, p])

        # === PLOT ===
        plot_name_base = f"{method}_Noise{noise_level}_{dirname}"
        plot_png = os.path.join(results_dir, plot_name_base + ".png")
        plot_pdf = os.path.join(results_dir, plot_name_base + ".pdf")

        plt.figure(figsize=(8, 8))
        sns.set(style="white", context="talk")
        ax = sns.regplot(
            x=true_vals_flat,
            y=pred_vals_flat,
            scatter_kws={'s': 30, 'alpha': 0.5, 'color': '#4C72B0'},
            line_kws={'color': 'crimson', 'lw': 2}
        )
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_xlabel('Actual Fraction (%)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Predicted Fraction (%)', fontsize=14, fontweight='bold')
        ax.set_title(
            f'{method} — Noise {noise_level}\n{dirname}\nPearson r = {r:.2f}, R² = {r2:.2f}, p = {p:.2e}',
            fontsize=14, fontweight='bold'
        )
        ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
        sns.despine()
        plt.tight_layout()
        plt.savefig(plot_png, dpi=300, bbox_inches="tight")
        plt.savefig(plot_pdf, dpi=300, bbox_inches="tight")
        plt.close()

    except Exception as e:
        print(f" Error processing {file_path}: {e}\n")

# === SAVE SUMMARY TABLE ===
if all_results:
    results_df = pd.DataFrame(
        all_results,
        columns=["ProportionsFile", "InputDir", "Method", "NoiseLevel", "Pearson_r", "R_squared", "p_value"]
    )
    results_df["Matrix"] = results_df["InputDir"].apply(lambda x: "_".join(x.split("_")[0:4]))
    results_df = results_df[[
        "ProportionsFile", "Matrix", "Method", "NoiseLevel", "Pearson_r", "R_squared", "p_value"
    ]]

    results_csv = os.path.join(results_dir, "All_Pearson_Summary.csv")
    results_df.to_csv(results_csv, sep='\t', index=False)
    print(f"\n Saved Pearson summary: {results_csv}")
else:
    print("\n⚠️ No valid results to save.")


Found 63 deconvolution result files.

TSP-BDa_Outer_300_1500_10_Random_v2C-Noisy_phi100_All-Counts | CIBERSORTx | Noise=100 — Pearson r = 0.6756, p = 0.00e+00
TSP-BDa_Outer_300_1500_10_Random_v2C-Noisy_phi100_All-Counts | NNLS | Noise=100 — Pearson r = 0.3114, p = 0.00e+00
TSP-BDa_Outer_300_1500_10_Random_v2C-Noisy_phi100_All-Counts | QP | Noise=100 — Pearson r = 0.3275, p = 0.00e+00
TSP-BDa_Outer_300_1500_10_Random_v2C-Noisy_phi100_All-Counts | BayesPrism | Noise=100 — Pearson r = 0.7522, p = 0.00e+00
TSP-BDa_Outer_300_1500_10_Random_v2C-Noisy_phi100_All-Counts | ReDeconv | Noise=100 — Pearson r = 0.7564, p = 0.00e+00
TSP-BDa_Outer_300_1500_10_Random_v2C-Noisy_phi100_All-Counts | MuSiC | Noise=100 — Pearson r = 0.6347, p = 0.00e+00
TSP-BDa_Outer_300_1500_10_Random_v2C-Noisy_phi100_All-Counts | nuSVR | Noise=100 — Pearson r = 0.5266, p = 0.00e+00
TSP-BDa_Outer_300_1500_10_Random_v2C-Noisy_phi10_All-Counts | CIBERSORTx | Noise=10 — Pearson r = 0.7628, p = 0.00e+00
TSP-BDa_Outer_300_1500

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# === LOAD DATA ===
file_path = 'COO-Decon-Noise/Results-Pearson-PredictedVSActual/All_Pearson_Summary.csv'
df = pd.read_csv(file_path, sep='\t')

# === ENSURE NUMERIC NOISE LEVEL FOR SORTING ===
df['NoiseLevelNumeric'] = pd.to_numeric(df['NoiseLevel'], errors='coerce')

# === GROUP & AVERAGE (if multiple runs per noise/method exist) ===
agg_df = (
    df.groupby(['Method', 'NoiseLevelNumeric'], as_index=False)['Pearson_r']
    .mean()
)

# === PIVOT TO METHOD × NOISE LEVEL MATRIX ===
pivot_df = agg_df.pivot_table(
    index='Method',
    columns='NoiseLevelNumeric',
    values='Pearson_r'
).sort_index(axis=1)

# === REORDER ROWS ===
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
pivot_df = pivot_df.reindex(method_order)

# === PLOT HEATMAP ===
fig, ax = plt.subplots(figsize=(6, 4), dpi=600, constrained_layout=True)
ax = sns.heatmap(
    pivot_df,
    cmap='coolwarm',             
#    annot=True,
#    fmt=".1f",
#    annot_kws={"size": 10},
    cbar_kws={'label': 'Pearson r', 'shrink': 0.9},
    linewidths=0,
    linecolor='white',
    center=0 
)

# === AXIS & COLORBAR STYLING ===
ax.tick_params(axis='x', which='both', bottom=True, top=False, labelbottom=True, length=4, width=0.6)
ax.tick_params(axis='y', which='both', left=True, right=False, labelleft=True, length=4, width=0.6)

cbar = ax.collections[0].colorbar
cbar.ax.set_ylabel("Pearson r", fontsize=13, rotation=270, labelpad=20) 
cbar.ax.tick_params(labelsize=11, width=0.8, length=4)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === LABELS ===
plt.ylabel('Deconvolution Tool', fontsize=14, labelpad=10)
plt.xlabel('Noise Level', fontsize=14, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

# === SAVE & SHOW ===
out_path = 'COO-Decon-Noise/Results-Pearson-PredictedVSActual/Pearson_Heatmap_NoiseLevels_Clean.svg'
fig.savefig(out_path)
plt.close(fig)